In [72]:
import numpy as np
import pandas as pd
import pm4py
import json
from openai import OpenAI
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from mppn.eventlog import EventLog
from mppn.clustering import Clustering
import csv


In [73]:
log = pm4py.read_xes("./datasets/BPIC15GroundTruth.xes", return_legacy_log_object=True)

parsing log, completed traces ::   0%|          | 0/5649 [00:00<?, ?it/s]

In [74]:
# log = pm4py.read_xes("./datasets/BPIC2019_sampled.xes",return_legacy_log_object=True)

In [75]:
def trace_to_text(trace, event_attributes=None):
      event_attributes = event_attributes or []
      events = []
      for event in trace:
          activity = event["concept:name"].replace(" ","_")
          attrs = []
          for a in event_attributes:
              attrs = [f"{a}:{event[a]}" for a in event_attributes if event.get(a) is not None]

          if attrs:
              events.append(f"{activity}[{', '.join(attrs)}]")
          else:
              events.append(str(activity))

      header = f"Trace: "
      return header + " -> ".join(events)


In [76]:
trace_texts = []
event_attributes_2015 = ["org:resource","monitoringResource"]
event_attributes_2019= ["org:resource"]
for trace in log:
    trace_texts.append(trace_to_text(trace, event_attributes_2015))

txt_file = "2015trace_texts.txt"
# txt_file = "2019trace_texts.txt"
with open(txt_file, "w", encoding="utf-8") as f:
    for i, t in enumerate(trace_texts):
        f.write(f"{i},{t}\n")

print("Trace texts saved:", txt_file)


Trace texts saved: 2015trace_texts.txt


In [77]:
txt_file = "2015trace_texts.txt"
# txt_file = "2019trace_texts.txt"
with open(txt_file, "r", encoding="utf-8") as f:
    trace_texts = [line.strip().split(",",1)[1] for line in f if line.strip()]

print("Loaded traces:", len(trace_texts))


Loaded traces: 5649


In [78]:
import time
import os

client = OpenAI(
    api_key="your_api_key",
    base_url="your_url"
)
batch_size = 100
vectors = []
embedding_failed = False

for i in range(0, len(trace_texts), batch_size):

    batch = trace_texts[i:i+batch_size]

    for attempt in range(3):
        try:
            response = client.embeddings.create(
                model="text-embedding-3-small",
                input=batch
            )
            for item in response.data:
                vectors.append(item.embedding)
            break
        except Exception as e:
            if attempt == 2:
                print("Embedding failed, fallback to random vectors:", e)
                embedding_failed = True
            else:
                time.sleep(2 ** attempt)
    if embedding_failed:
        break

if embedding_failed:
    # fallback: random vectors to allow pipeline to continue
    rng = np.random.default_rng(42)
    vectors = rng.normal(size=(len(trace_texts), 1536)).tolist()

print("Embedding vectors:", len(vectors))
vectors = normalize(np.array(vectors))

out_file = "2015trace_vectors_1536.csv"
# out_file = "2019trace_vectors_1536.csv"
with open(out_file, "w", newline="") as f:
    writer = csv.writer(f)

    # 写 header（维度数，非样本数）
    dim = vectors.shape[1]
    header = ["trace_id"] + [f"v{i}" for i in range(dim)]
    writer.writerow(header)

    # 写数据
    for i, vec in enumerate(vectors):
        writer.writerow([i] + list(vec))

print("Vectors saved to", out_file)



Embedding vectors: 5649
Vectors saved to 2015trace_vectors_1536.csv


In [79]:
trace_llm_embedding_out_file = "2015trace_vectors_1536.csv"
# trace_llm_embedding_out_file = "2019trace_vectors_1536.csv"
vectors_df = pd.read_csv(trace_llm_embedding_out_file)

# 仅保留向量列，避免 trace_id 和缺失列影响 PCA
if "trace_id" in vectors_df.columns:
    vectors_df = vectors_df.drop(columns=["trace_id"])

vectors_df = vectors_df.apply(pd.to_numeric, errors="coerce")
# 兼容旧文件：去掉全空列
vectors_df = vectors_df.dropna(axis=1, how="all")

n_components = 128
if len(vectors_df) <= n_components:
    n_components = max(2, len(vectors_df)-1)

pca = PCA(n_components=n_components)
vectors_reduced = pca.fit_transform(vectors_df.values)

print("Reduced dimension:", len(vectors_reduced[0]))



/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:335: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:336: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:33

Reduced dimension: 128


/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: divide by zero encountered in matmul
  B = Q.T @ M
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: overflow encountered in matmul
  B = Q.T @ M
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:533: RuntimeWarning: invalid value encountered in matmul
  B = Q.T @ M
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uh

In [80]:
out_file = "trace_vectors_128.csv"
# out_file = "2019trace_vectors_128.csv"
with open(out_file, "w", newline="") as f:
    writer = csv.writer(f)

    # 写 header
    header = ["trace_id"] + [f"v{i}" for i in range(128)]
    writer.writerow(header)

    # 写数据
    for i, vec in enumerate(vectors_reduced):
        writer.writerow([i] + list(vec))

print("Vectors saved to", out_file)


Vectors saved to trace_vectors_128.csv


In [81]:
# event log configuration
event_log_path = "datasets"
file_name = "BPIC15GroundTruth.xes"

case_attributes = []  # auto-detect attributes
event_attributes = ["concept:name", "org:resource","monitoringResource"]  # use activity name and user
true_cluster_label = "cluster:label"

# file_name = "BPIC2019_sampled.xes"
#
# case_attributes = ["Item Type","Document Type"]  # auto-detect attributes
# event_attributes = ["concept:name", "org:resource"]  # use activity name and user
# true_cluster_label = "Item Category"

# load file
event_log = EventLog(file_name, case_attributes=case_attributes, event_attributes=event_attributes,
                     true_cluster_label=true_cluster_label)
event_log.load(event_log_path + "/" + file_name, False)
event_log.preprocess()


parsing log, completed traces ::   0%|          | 0/5649 [00:00<?, ?it/s]

In [82]:
import pandas as pd

# load vectors for clustering
out_file = "trace_vectors_128.csv"
# out_file = "2019trace_vectors_128.csv"
feature_vector = pd.read_csv(out_file)
X = feature_vector.drop(columns=["trace_id"]).values
labels = event_log.true_cluster_labels
n_clusters = len(set(labels))

cluster_analysis = Clustering(event_log)
cluster_analysis.cluster(X, "k_means", n_clusters, "cosine")
cluster_result = cluster_analysis.evaluate()
print("Normalized Mutual Information:", cluster_result[1])
print("F1-BCubed:", cluster_result[2])


Normalized Mutual Information: 0.13946440145322833
F1-BCubed: 0.30493641339568095


/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/chengyy/tmpCode/traceEmbedding/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.

In [83]:
from pm4py.objects.log.obj import EventLog as PMEventLog
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.algo.evaluation.replay_fitness import algorithm as fitness_algo
from pm4py.algo.evaluation.precision import algorithm as precision_algo

# group traces by predicted label (cluster)
pred_labels = cluster_analysis.pred_labels
sublogs = {}
for idx, trace in enumerate(log):
    lbl = pred_labels[idx]
    if lbl not in sublogs:
        sublogs[lbl] = PMEventLog()
    sublogs[lbl].append(trace)

print('num_sublogs:', len(sublogs))

results = []

for lbl, sublog in sorted(sublogs.items(), key=lambda x: x[0]):

    tree = inductive_miner.apply(sublog, variant=inductive_miner.Variants.IMf)

    net, im, fm = pt_converter.apply(tree)

    fitness = fitness_algo.apply(
        sublog, net, im, fm,
        variant=fitness_algo.Variants.TOKEN_BASED
    )

    precision = precision_algo.apply(
        sublog, net, im, fm,
        variant=precision_algo.Variants.ETCONFORMANCE_TOKEN
    )

    fitness_value = fitness["log_fitness"]
    precision_value = precision

    if fitness_value + precision_value > 0:
        f1 = 2 * fitness_value * precision_value / (fitness_value + precision_value)
    else:
        f1 = 0

    results.append((lbl, len(sublog), fitness_value, precision_value, f1))

    print(
        f"Cluster {lbl} size={len(sublog)} "
        f"fitness={fitness_value:.4f} "
        f"precision={precision_value:.4f} "
        f"f1={f1:.4f}"
    )
print(results)

# 按 cluster 大小加权计算 F1
_total = sum(size for _, size, _, _, _ in results)
if _total > 0:
    weighted_f1 = sum(f1 * size for _, size, _, _, f1 in results) / _total
else:
    weighted_f1 = 0

print("Weighted F1-score:", round(weighted_f1, 5))


num_sublogs: 5


replaying log with TBR, completed traces ::   0%|          | 0/219 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1057 [00:00<?, ?it/s]

Cluster 0 size=731 fitness=1.0000 precision=0.1703 f1=0.2910


replaying log with TBR, completed traces ::   0%|          | 0/464 [00:00<?, ?it/s]

...

replaying log with TBR, completed traces ::   0%|          | 0/3928 [00:00<?, ?it/s]

Cluster 3 size=767 fitness=1.0000 precision=0.1402 f1=0.2459


replaying log with TBR, completed traces ::   0%|          | 0/418 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1606 [00:00<?, ?it/s]

Cluster 4 size=1274 fitness=1.0000 precision=0.2526 f1=0.4033
[(np.int32(0), 731, 1.0, 0.17030565041536483, 0.29104473750924803), (np.int32(1), 1369, 1.0, 0.17174180427687658, 0.2931393309516076), (np.int32(2), 1508, 0.9845726773864356, 0.15004745582877776, 0.2604089615468526), (np.int32(3), 767, 1.0, 0.14018140660860812, 0.24589316365992697), (np.int32(4), 1274, 1.0, 0.2525881536880864, 0.40330599158929087)]
Weighted F1-score: 0.30256
